In [20]:
import pandas as pd
import openai
from qdrant_client import QdrantClient
from dotenv import load_dotenv
from langsmith import traceable, get_current_run_tree # for cost calculation
load_dotenv("../../.env")


True

In [5]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

In [27]:
@traceable(name="get_embedding", run_type = "embedding",metadata = {"ls_model_name": "text-embedding-3-small", "ls_provider": "openai"})
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens
        }
    return response.data[0].embedding

@traceable(name="retrieve_from_qdrant", run_type = "retriever")
def retrieve_from_qdrant(query, k=5):
    response = qdrant_client.query_points(
        collection_name="amazon-electronics-items-collection-01",
        query=get_embedding(query),
        limit=k
    )
    retrieved_context_ids = []
    retrived_context = []
    similarity_scores = []
    retrived_context_ratings = []
    for point in response.points:
        retrieved_context_ids.append(point.payload['parent_asin'])
        retrived_context.append(point.payload['preprocessed_description'])
        similarity_scores.append(point.score)
        retrived_context_ratings.append(point.payload['average_rating'])
        
    return {"retrieved_context_ids": retrieved_context_ids,
            "retrived_context": retrived_context,
            "similarity_scores": similarity_scores,
            "retrived_context_ratings": retrived_context_ratings}





### format retrieved function


In [22]:
@traceable(name="process_context", run_type = "prompt")
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrived_context'], context['retrived_context_ratings']):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n"
    return formatted_context

In [6]:
retrieved_context = retrieve_from_qdrant("do you have a usd connectable fan for hot weather?", k=10)
preprocessed_context = process_context(retrieved_context)

In [9]:
print(preprocessed_context)

- ID: B016KQ7IGG, Rating: 3.8, Description: ThreeH USB Fan Laptop Cooling Pad with 3 Fans & Blue LED Lights for Laptop PS3 / PS4 / PS Slim H-UF01 Size: Fits 10-15" laptops - Works fine for laptops either smaller or larger than its size. USB port: Powered through USB port,anti-slide. Design: 3 Blue LED lights built into the pad. Good heat dissipation: 3 Fans to avoid overheating,keep your computer good performance. Anti-slide: Round rubber pad with a high friction coefficient on four corner of the fan around to ensure that the notebook will not be damaged by sliding.
- ID: B08F9T8LLQ, Rating: 4.7, Description: KMC 6-Outlet Surge Tap, 2 USB Ports (3.4A), 980 Joules Surge Protector, White (2 Pack) Multi Outlet: 6 AC outlets (15A/125V/1875W) can meet the need of 6 devices at the same time. With built-in surge protector (980J), it can ensure safety. When the “Protected” indicator LED light turns on, it shows your devices are protected against electrical spikes at the maximum. Dual Smart USB

### prompt template

In [23]:
@traceable(name="build_prompt", run_type = "prompt")
def build_prompt(preprocessed_context, question):
    prompt = f"""
    You are a helpful shopping assistant that can answer questions about the product in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available product.
    - Do not use markdown formatting.

    Here is the context:
    {preprocessed_context}
    Here is the question:
    {question}
    """
    return prompt

In [13]:
prompt = build_prompt(preprocessed_context, "do you have a usd connectable fan for hot weather?")
print(prompt)


    You are a helpful shopping assistant that can answer questions about the product in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available product.
    - Do not use markdown formatting.

    Here is the context:
    - ID: B016KQ7IGG, Rating: 3.8, Description: ThreeH USB Fan Laptop Cooling Pad with 3 Fans & Blue LED Lights for Laptop PS3 / PS4 / PS Slim H-UF01 Size: Fits 10-15" laptops - Works fine for laptops either smaller or larger than its size. USB port: Powered through USB port,anti-slide. Design: 3 Blue LED lights built into the pad. Good heat dissipation: 3 Fans to avoid overheating,keep your computer good performance. Anti-slide: Round rubber pad with a high friction coefficient on four corner of the fan around to ensure that the notebook will not be damaged by sliding.
- ID: B08F9T8LLQ, Rating: 4.7, Description: KMC 6-Outl

### generate answer function

In [28]:
@traceable(name="generate_answer", run_type = "llm", metadata = {"ls_model_name": "gpt-5.4-nano", "ls_provider": "openai"})
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning_effort="none"
    )
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    return response.choices[0].message.content

In [16]:
generate_answer(prompt)

'Yes. We have a USB-powered laptop cooling fan: the ThreeH USB Fan Laptop Cooling Pad with 3 Fans & Blue LED Lights (fits 10–15" laptops, powered through a USB port, with 3 fans for heat dissipation). It also has anti-slide rubber pads to help keep your laptop stable while it’s running.'

### combine rag pipeline


In [29]:
@traceable(name="rag_pipeline")
def rag_pipeline(question, k=5):
    retrieved_context = retrieve_from_qdrant(question, k=k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    return answer


In [30]:
print(rag_pipeline("do you have a usd connectable fan for hot weather?"))

Yes. The available USB fan laptop cooling pad has 3 built-in fans with blue LED lights and is powered through a USB port. It’s designed to help avoid overheating and is sized to fit 10–15" laptops.


In [23]:
print(rag_pipeline("could you suggest some earphones with good ratings"))

Yes—based on the available products, here are a few earphones/headphone options with good ratings:

1) B075V4T5X2 (3.6 rating)  
Mini Invisible Bluetooth Earbud (pink5), Bluetooth V4.1, built-in mic, HD stereo sound, up to about 4 hours playtime.

2) B07FZ3G7SC (4.5 rating)  
Esinkin Wireless Audio Receiver (not earphones; it’s a wireless adapter for streaming to speakers/sound systems). Strong sound and up to about 15 meters range. (If you want earphones specifically, you may prefer other items.)

If you specifically mean Bluetooth earbuds/earphones (instead of receivers or cables), tell me whether you want Bluetooth earbuds or wired 3.5mm and I can narrow it down from what’s available.
